# SupplyForge AI 2.0: KAN-31 - Categorical Features Encoding



Tài liệu này thực hiện việc mã hóa các biến phân loại của sản phẩm và kho hàng thành các biểu diễn tương thích với mô hình học máy (LightGBM) mà không gây rò rỉ dữ liệu.



Các biến phân loại gồm: `size`, `type`, `brand`, `categoryId`, và `warehouse` được mã hóa thành các số nguyên deterministic (Label Encoding) hoặc kiểu dữ liệu `category` của Pandas hỗ trợ nguyên bản thuật toán LightGBM.


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **Mã hóa deterministic**: Xây dựng từ điển ánh xạ cho từng đặc trưng. Giá trị giống nhau luôn nhận cùng một mã số nguyên.

2. **Hỗ trợ nhãn ngoại lệ (Unknown Handling)**: Mỗi từ điển bổ sung nhãn đặc biệt `"Unknown"`. Khi gặp giá trị lạ trong quá trình inference, tự động gán về nhãn này.

3. **Xuất từ điển**: Lưu các ánh xạ thành các tệp JSON độc lập để tái sử dụng trong quá trình serving.

4. **Cập nhật dữ liệu**: Lưu trữ cột đã mã hóa vào danh mục tã bỉm `diaper_skus_attributes.csv` và lưới ngày `calendar_daily_sales_metadata.csv` (chuyển đổi thành kiểu `category`).


In [1]:
import pandas as pd
import numpy as np
import json
import os

PROJECT_ROOT = "/home/nguyen-dai-tung/supplyforge-ai"
SILVER_DIR = os.path.join(PROJECT_ROOT, "data", "silver")
GOLD_DIR = os.path.join(PROJECT_ROOT, "data", "gold")

items_path = os.path.join(SILVER_DIR, "order_items_clean.parquet")
product_path = os.path.join(SILVER_DIR, "products_clean.parquet")
diaper_path = os.path.join(GOLD_DIR, "diaper_skus_attributes.parquet")
if not os.path.exists(diaper_path):
    diaper_path = os.path.join(GOLD_DIR, "diaper_skus_attributes.csv")
grid_path = os.path.join(GOLD_DIR, "calendar_daily_sales_metadata.parquet")
if not os.path.exists(grid_path):
    grid_path = os.path.join(GOLD_DIR, "calendar_daily_sales_metadata.csv")

print("Đang tải dữ liệu từ Silver & Gold Layer...")
df_items = pd.read_parquet(items_path)
df_products = pd.read_parquet(product_path)
df_diapers = pd.read_parquet(diaper_path) if diaper_path.endswith(".parquet") else pd.read_csv(diaper_path)
df_grid = pd.read_parquet(grid_path) if grid_path.endswith(".parquet") else pd.read_csv(grid_path)
print(f"Loaded items: {len(df_items):,}, products: {len(df_products):,}, grid: {len(df_grid):,}")


Đang tải dữ liệu từ Silver & Gold Layer...


Loaded items: 559,254, products: 1,335, grid: 1,597,060


In [2]:
print("=== XÂY DỰNG BẢNG ÁNH XẠ KHO HÀNG (WAREHOUSE) ===")
print("Đang xây dựng ánh xạ kho hàng từ order_items sạch...")
df_wh = df_items[df_items["warehouse"].str.strip() != ""].drop_duplicates(subset=["itemId"])
item_wh_mapping = dict(zip(df_wh["itemId"].astype(str), df_wh["warehouse"]))
print(f"Số lượng itemId đã ánh xạ kho hàng: {len(item_wh_mapping):,}")


=== XÂY DỰNG BẢNG ÁNH XẠ KHO HÀNG (WAREHOUSE) ===
Đang xây dựng ánh xạ kho hàng từ order_items sạch...
Số lượng itemId đã ánh xạ kho hàng: 713


In [3]:
print("=== TẠO TỪ ĐIỂN MÃ HÓA DETERMINISTIC CATEGORICAL ENCODING ===")
categorical_cols_values = {
    "size": sorted(list(df_diapers["size"].dropna().astype(str).unique())),
    "type": sorted(list(df_diapers["type"].dropna().astype(str).unique())),
    "brand": sorted(list(df_products["brand"].dropna().astype(str).unique())),
    "categoryId": sorted([str(c) for c in df_products["categoryId"].dropna().unique()]),
    "warehouse": sorted(list(set(item_wh_mapping.values())))
}

encoding_dicts = {}
for col, vals in categorical_cols_values.items():
    if "Unknown" not in vals:
        vals.append("Unknown")
    mapping = {val: idx for idx, val in enumerate(vals)}
    encoding_dicts[col] = mapping
    json_path = os.path.join(GOLD_DIR, f"{col}_mapping.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(mapping, f, indent=4, ensure_ascii=False)
    print(f"Đã xuất từ điển mã hóa {col} tại: {json_path}")

print("Đang áp dụng mã hóa vào df_diapers và df_grid...")
if "brand" not in df_diapers.columns:
    df_diapers["brand"] = "Unknown"
if "categoryId" not in df_diapers.columns:
    df_diapers["categoryId"] = "Unknown"
df_diapers["warehouse"] = df_diapers["itemId"].astype(str).map(item_wh_mapping).fillna("Unknown")

for col in ["size", "type", "brand", "categoryId", "warehouse"]:
    mapping = encoding_dicts[col]
    unknown_idx = mapping.get("Unknown", 0)
    df_diapers[f"{col}_encoded"] = df_diapers[col].astype(str).map(mapping).fillna(unknown_idx).astype("int32")

df_grid["warehouse"] = df_grid["itemId"].astype(str).map(item_wh_mapping).fillna("Unknown")
for col in ["brand", "categoryId", "warehouse"]:
    mapping = encoding_dicts[col]
    unknown_idx = mapping.get("Unknown", 0)
    df_grid[f"{col}_encoded"] = df_grid[col].astype(str).map(mapping).fillna(unknown_idx).astype("category")


=== TẠO TỪ ĐIỂN MÃ HÓA DETERMINISTIC CATEGORICAL ENCODING ===
Đã xuất từ điển mã hóa size tại: /home/nguyen-dai-tung/supplyforge-ai/data/gold/size_mapping.json
Đã xuất từ điển mã hóa type tại: /home/nguyen-dai-tung/supplyforge-ai/data/gold/type_mapping.json
Đã xuất từ điển mã hóa brand tại: /home/nguyen-dai-tung/supplyforge-ai/data/gold/brand_mapping.json
Đã xuất từ điển mã hóa categoryId tại: /home/nguyen-dai-tung/supplyforge-ai/data/gold/categoryId_mapping.json
Đã xuất từ điển mã hóa warehouse tại: /home/nguyen-dai-tung/supplyforge-ai/data/gold/warehouse_mapping.json
Đang áp dụng mã hóa vào df_diapers và df_grid...


In [4]:
df_diapers.to_parquet(os.path.join(GOLD_DIR, "diaper_skus_attributes.parquet"), index=False)
df_diapers.to_csv(os.path.join(GOLD_DIR, "diaper_skus_attributes.csv"), index=False)
print("Đã lưu thuộc tính bỉm mã hóa vào Gold Layer.")


Đã lưu thuộc tính bỉm mã hóa vào Gold Layer.


In [5]:
df_grid.to_parquet(os.path.join(GOLD_DIR, "calendar_daily_sales_metadata.parquet"), index=False)
df_grid.to_csv(os.path.join(GOLD_DIR, "calendar_daily_sales_metadata.csv"), index=False)
print("Đã cập nhật lưới ngày mã hóa vào Gold Layer.")


Đã cập nhật lưới ngày mã hóa vào Gold Layer.


In [6]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===\n")







total_skus = len(df_diapers)



null_sizes = df_diapers['size'].isnull().sum()



null_types = df_diapers['type'].isnull().sum()







# Đảm bảo không có giá trị khuyết thiếu trong các cột mã hóa mới sinh ra



for col in ['size', 'type', 'brand', 'categoryId', 'warehouse']:



    assert df_diapers[f"{col}_encoded"].isnull().sum() == 0, f"Lỗi: Phát hiện Null ở đặc trưng mã hóa {col}!"



print("1. Đảm bảo các cột đặc trưng mã hóa mới không chứa giá trị Null: Đạt")







# Đảm bảo các giá trị số nguyên mã hóa là deterministic và hợp lệ



for col in ['size', 'type', 'brand', 'categoryId', 'warehouse']:



    mapping = encoding_dicts[col]



    assert df_diapers[f"{col}_encoded"].isin(mapping.values()).all(), f"Lỗi: Giá trị mã hóa của {col} không hợp lệ!"



print("2. Tính hợp lệ của mã hóa số nguyên: Đạt")







# Đảm bảo định dạng kiểu dữ liệu trên lưới hỗ trợ nguyên bản LightGBM



for col in ['brand', 'categoryId', 'warehouse']:



    assert df_grid[f"{col}_encoded"].dtype.name == 'category', f"Lỗi: Cột {col}_encoded không phải kiểu category!"



print("3. Định dạng Pandas category dtype tương thích với LightGBM: Đạt")







# In ngẫu nhiên 30 bản ghi thuộc tính tã bỉm để kiểm thử thủ công



print("\n4. In ngẫu nhiên 30 SKU đã mã hóa số hóa để kiểm định trực quan:")



print(df_diapers[['skuName', 'size', 'size_encoded', 'type', 'type_encoded', 'brand', 'brand_encoded', 'warehouse', 'warehouse_encoded']].sample(30, random_state=42).to_string())



=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===

1. Đảm bảo các cột đặc trưng mã hóa mới không chứa giá trị Null: Đạt
2. Tính hợp lệ của mã hóa số nguyên: Đạt
3. Định dạng Pandas category dtype tương thích với LightGBM: Đạt

4. In ngẫu nhiên 30 SKU đã mã hóa số hóa để kiểm định trực quan:
                 skuName       size  size_encoded   type  type_encoded    brand  brand_encoded     warehouse  warehouse_encoded
40       Quần L88 9-14kg          L             1  PANTS             0  Unknown             83  Kho gia dụng                  2
22  L44 Quần Gái Xanh TC  FREE_SIZE             0   TAPE             1  Unknown             83       Unknown                  4
55            L44+4 Quần  FREE_SIZE             0  PANTS             0  Unknown             83  Kho Gia dụng                  0
72   Quà tặng ngẫu nhiên  FREE_SIZE             0  PANTS             0  Unknown             83       Kho HVN                  1
0           Quần XL 38+6  FREE_SIZE             0   TAPE        